# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, in accordance with the Croissant schema and using entity `@id` notation throughout.

### Dataset Source
The dataset source is provided as a Croissant schema JSON-LD file:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print a summary from metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")

## 2. Data Overview
Let's explore available record sets, their `@id`, fields, and field `@id`.

You can inspect available record sets directly from the dataset's metadata. Here we print their `@id` and available fields (by their `@id` as well).

In [ ]:
# List all record sets (`cr:RecordSet` entries) and their fields via metadata
from pprint import pprint

# Most Croissant datasets expose recordSet as a list of objects; fallback to the attribute in case it's a single object.
if hasattr(metadata, 'record_set') and metadata.record_set:
    record_sets = metadata.record_set
else:
    # As per Croissant, use 'recordSet' if 'record_set' not found
    record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for i, rs in enumerate(record_sets):
        print(f"Record set #{i+1} @id: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields and their @id:")
        for field in fields:
            # Each field is an object or dict with '@id'
            field_id = field['@id'] if isinstance(field, dict) else getattr(field, '@id', '(unknown)')
            print(f"    {field_id}")
        print()
    pprint(record_sets[0])

# For demonstration, collect all record set @ids for later loading
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We reference the record set `@id` and field `@id` as above.

In [ ]:
if not record_set_ids:
    print("No available record sets to extract records from.")
else:
    record_set_to_use = record_set_ids[0]  # Just the first for demo
    # Extract records from the chosen record set by @id
    records_iter = dataset.records(record_set=record_set_to_use)
    records_list = list(records_iter)
    if not records_list:
        print(f"No records found for record set @id: {record_set_to_use}")
    else:
        df = pd.DataFrame(records_list)
        print(f"Loaded DataFrame for record set @id: {record_set_to_use}")
        print(df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group data using field `@id`s.

Below, choose one numeric field and one categorical/grouping field from those printed in the prior cell, referencing only by `@id`.

> _Update the variable assignments below to match the actual field @ids in your target record set._

In [ ]:
# Example variable assignments (set these IDs accordingly from the prior overview):
numeric_field_id = 'abundance_norm_sample_1'   # example @id, replace with your numeric field @id if different
group_field_id = 'accession'   # example @id, replace with a useful categorical field @id

if not record_set_ids:
    print("No available record sets to process.")
elif 'df' not in globals():
    print("No data loaded; please rerun extraction cell above.")
elif numeric_field_id not in df.columns:
    print(f"Numeric field @id '{numeric_field_id}' not found in DataFrame columns: {df.columns.tolist()}")
else:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualizing the distribution of the numeric field and its grouping.

The following plots use the field `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in globals() and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field exists and cardinality is reasonable
    if group_field_id in filtered_df.columns and filtered_df[group_field_id].nunique() < 30:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Run the previous cell to load and process data first.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using its Croissant schema and explored its data using field and record set `@id` references. The use of the mlcroissant library enables standards-based, reproducible data loading and processing. Further analysis can be conducted by exploring other record sets or fields as needed.